# Version 2: Scraping Data from PDF Report

### Report: Individual Business Seach <a href = "Data/ibs.pdf"> IBS </a>

Data source: PDF Report

Output: Excel file containing the tables of data

## IBS 
- Contains 12 tables in the entire report
    > REQUEST CRITERIA
    
    > SEARCH RECORD

    > CURRENT BUSINESS

    > PREVIOUS BUSINESS

    > CURRENT BUSINESS LITIGATION TRACE

    > BUSINESS LITIGATION SEARCH – WRIT OF SUMMONS/ ORIGINATING CLAIM

    > BANKRUPTCY RECORD(S)

    > INDIVIDUAL LITIGATION SEARCH – WRIT OF SUMMONS/ ORIGINATING CLAIM

    > ACCREDITATIONS – CURRENT BUSINESS

    > FINANCIAL RELATED SEARCH COUNT

    > COMMERCIAL RELATED SEARCH COUNT

    > LEGAL RELATED SEARCH COUNT

Import Libraries and Dependencies

In [1]:
import pdfplumber
import pandas as pd
import numpy as np
import itertools
import re
import joblib
import sys

sys.path.append('../../Litigation Clustering/')

from litigation_clustering import label_litigation_main_cause, label_litigation_case_status, label_litigation_party_status

In [2]:
# Open the PDF file using pdfplumber and read all lines into a single list
dataExtraction = []
x_tolereance = 8
with pdfplumber.open("../../Dataset/ibs_data/ibs_sample_2.pdf") as pdf:

    # Iterating through each page of the pdf report
    for page in range(len(pdf.pages)):
        page_section = pdf.pages[page].find_tables()
        tableCount = 0
        while True:
            # If there are more tables to be read for the given page, the try block will be executed
            try:
                table_content = page_section[tableCount].extract(x_tolerance = x_tolereance)
                dataExtraction.append(
                    [[ele for ele in table_content_data if ele not in [None, '']] for table_content_data in table_content]
                )
                tableCount += 1
            except:
                break

Flattening the array of data into one singular list

In [3]:
compiledList = [ele for rowData in dataExtraction for ele in rowData]

Initialising Table Dictionary

In [4]:
# This dictionary will be used to store all tables in the given report
# Key: Table name
# Value: List containing the data specific to the corresponding table
tablesDict = {
   'REQUEST CRITERIA':                                                                          [],
   'SEARCH RECORD':                                                                             [],
   'CURRENT BUSINESS':                                                                          [],
   'PREVIOUS BUSINESS':                                                                         [],
   'CURRENT BUSINESS LITIGATION TRACE\nNote: Available Information As Per Experian Database':   [],
   'BUSINESS LITIGATION SEARCH – WRIT OF SUMMONS/ ORIGINATING CLAIM':                           [],
   'BANKRUPTCY RECORD(S)':                                                                      [],
   'INDIVIDUAL LITIGATION SEARCH – WRIT OF SUMMONS/ ORIGINATING CLAIM':                         [],
   'ACCREDITATIONS – CURRENT BUSINESS':                                                         [],
   'FINANCIAL RELATED SEARCH COUNT':                                                            [],
   'COMMERCIAL RELATED SEARCH COUNT':                                                           [],
   'LEGAL RELATED SEARCH COUNT':                                                                [],
}

Filtering the Respective Tabular Data

In [5]:
tableList = []
currentKey = None

# Iterating through each element in the singular list that we have compiled into
for rowData in compiledList[:-1]:

    # If there are no data or element present in the given row, we will simply continue
    if len(rowData) == 0:
        continue

    # If we hit an element that is one of the headers in our table dictionary, that would signal the
    # start of the respective table, and hence would need to compile the data that we have accumulated in tableList
    # and append the result in the preceding table
    elif rowData[0] in list(tablesDict.keys()):
        currentTable = currentKey
        currentKey = rowData[0]

        # If there are elements in tableList, we will append the entire result to the previous table 
        # and reinitialise the tableList
        if len(tableList) > 0:
            tablesDict[currentTable] = tableList
            tableList = []

    # If none of the previous 2 conditions were met, we would simply append the data rows into the tableList
    else:
        tableList.append(rowData)

# Add the last table to the tablesList
tablesDict['LEGAL RELATED SEARCH COUNT'] = tableList

Processing Table Data 

Table 1: REQUEST CRITERIA

In [6]:
# Remove the information row, which is simply the first element in the array
requestCriteriaArr = np.array(tablesDict['REQUEST CRITERIA'][1:])
requestCriteriaHeaders = [h.replace(':', '').replace('*','') for h in requestCriteriaArr[:,0]]
requestCriteriaData = requestCriteriaArr[:,1]
REQUESTCRITERIA = pd.DataFrame([requestCriteriaData], columns=requestCriteriaHeaders)
REQUESTCRITERIA

,Date of Request,Name of Requestor,Requested Individual Name,Requested ID Number,File Reference Number
0,09/01/20XX,SUBSCRIBER,TAN AH KOW,S123XXXXE,IBS SAMPLE


In [7]:
columnNames = [
    'Date of Request', 
    'Name of Requestor', 
    'Requested Individual Name', 
    'Requested ID Number', 
    'File Reference Number', 
]
for actual, expected in itertools.zip_longest(REQUESTCRITERIA.columns, columnNames):
    assert actual == expected, f'REQUEST CRITERIA has column {actual} but should be {expected}'

In [8]:
REQUESTCRITERIA.columns = [
    'request_date', 
    'requestor_name', 
    'requested_individual_name', 
    'requested_id_no', 
    'request_id', 
]
REQUESTCRITERIA

,request_date,requestor_name,requested_individual_name,requested_id_no,request_id
0,09/01/20XX,SUBSCRIBER,TAN AH KOW,S123XXXXE,IBS SAMPLE


Table 2: SEARCH RECORD

In [9]:
# Collect the data in the first 3 rows of the array
searchRecordArr = np.array(tablesDict['SEARCH RECORD'][:3])
searchRecordHeaders = [h.replace(':', '').replace('\n','') for h in searchRecordArr[:,0]]
searchRecordData = searchRecordArr[:,1]
SEARCHRECORD = pd.DataFrame([searchRecordData], columns=searchRecordHeaders)
SEARCHRECORD

,Individual Name(As per Experian database),ID Number,Available Address(As per Experian database)
0,1) TAN AH KOW\n2) MICHAEL TAN AH KOW\n3) TAN A...,S123XXXXE,123 ANG MO KIO #06-239\nSINGAPORE 123456


In [10]:
columnNames = [
    'Individual Name(As per Experian database)', 
    'ID Number', 
    'Available Address(As per Experian database)', 
]
for actual, expected in itertools.zip_longest(SEARCHRECORD.columns, columnNames):
    assert actual == expected, f'SEARCH RECORD has column {actual} but should be {expected}'

In [11]:
SEARCHRECORD.columns = [
    'individual_name', 
    'individual_id', 
    'available_address', 
]
SEARCHRECORD

,individual_name,individual_id,available_address
0,1) TAN AH KOW\n2) MICHAEL TAN AH KOW\n3) TAN A...,S123XXXXE,123 ANG MO KIO #06-239\nSINGAPORE 123456


Table 3: CURRENT BUSINESS

In [12]:
currentBusinessData = {}
index = 0
while index < len(tablesDict['CURRENT BUSINESS']):
    if tablesDict['CURRENT BUSINESS'][index] == ['No.', 'Name', 'Registration Number', 'Status', 'Position', 'Appointment Date']:
        for column, value in zip(tablesDict['CURRENT BUSINESS'][index][1:], tablesDict['CURRENT BUSINESS'][index + 1][1:]):
            if column in currentBusinessData:
                currentBusinessData[column].append(value)
            else:
                currentBusinessData[column] = [value]
        for i in range(4, 7):
            for j in range(2):
                column = tablesDict['CURRENT BUSINESS'][index + i][2 * j]
                if column in currentBusinessData:
                    currentBusinessData[column].append(tablesDict['CURRENT BUSINESS'][index + i][2 * j + 1])
                else:
                    currentBusinessData[column] = [tablesDict['CURRENT BUSINESS'][index + i][2 * j + 1]]
        index += 6
    index += 1
CURRENTBUSINESS = pd.DataFrame(currentBusinessData)
CURRENTBUSINESS

,Name,Registration Number,Status,Position,Appointment Date,Current Directors,Current Secretaries,Current Alternate Directors,Active Charges,Current Shareholders / Owners,Subsidiary/Associates
0,ABC HOLDINGS PTE LTD,1990XXXXH,LIVE,DIRECTOR\nSHAREHOLDER,23/01/19XX,2,0,0,2,2,0
1,WWW PTE LTD,20000XXXXW,LIVE,DIRECTOR\nSHAREHOLDER,01/02/20XX,3,1,0,1,3,0
2,T.A.K. SPECIALIST,5301XXXXJ,LIVE,OWNER,15/06/20XX,0,0,0,0,1,0


In [13]:
columnNames = [
    'Name', 
    'Registration Number', 
    'Status', 
    'Position',
    'Appointment Date',
    'Current Directors', 
    'Current Secretaries',
    'Current Alternate Directors', 
    'Active Charges',
    'Current Shareholders / Owners', 
    'Subsidiary/Associates', 
]
for actual, expected in itertools.zip_longest(CURRENTBUSINESS.columns, columnNames):
    assert actual == expected, f'CURRENT BUSINESS has column {actual} but should be {expected}'

In [14]:
CURRENTBUSINESS.columns = [
    'business_name', 
    'registration_no', 
    'status', 
    'position', 
    'appointment_date', 
    'current_directors_todate_count', 
    'current_secretaries_todate_count', 
    'current_alternate_directors_todate_count', 
    'active_charges_todate_count', 
    'current_shareholders_todate_count', 
    'subsidiary_todate_count', 
]
CURRENTBUSINESS

,business_name,registration_no,status,position,appointment_date,current_directors_todate_count,current_secretaries_todate_count,current_alternate_directors_todate_count,active_charges_todate_count,current_shareholders_todate_count,subsidiary_todate_count
0,ABC HOLDINGS PTE LTD,1990XXXXH,LIVE,DIRECTOR\nSHAREHOLDER,23/01/19XX,2,0,0,2,2,0
1,WWW PTE LTD,20000XXXXW,LIVE,DIRECTOR\nSHAREHOLDER,01/02/20XX,3,1,0,1,3,0
2,T.A.K. SPECIALIST,5301XXXXJ,LIVE,OWNER,15/06/20XX,0,0,0,0,1,0


Table 4: PREVIOUS BUSINESS

In [15]:
previousBusinessArr = np.array(tablesDict['PREVIOUS BUSINESS'])
previousBusinessHeaders = previousBusinessArr[0][1:]
previousBusinessData = [row[1:] for row in previousBusinessArr[1:]]
PREVIOUSBUSINESS = pd.DataFrame(previousBusinessData, columns=previousBusinessHeaders)
PREVIOUSBUSINESS

,Name,Registration Number,Status,Position,Cessation Date
0,ABCDE PTE LTD,19960XXXXC,LIVE,DIRECTOR,09/11/19XX
1,NETWORK DENKI,5021XXXXW,TERMINATION,OWNER,21/10/20XX
2,PIONEER INSTRUMENTS,5038XXXXC,CANCELLATION,OWNER,09/11/19XX


In [16]:
columnNames = [
    'Name', 
    'Registration Number', 
    'Status', 
    'Position', 
    'Cessation Date', 
]
for actual, expected in itertools.zip_longest(PREVIOUSBUSINESS.columns, columnNames):
    assert actual == expected, f'PREVIOUS BUSINESS has column {actual} but should be {expected}'

In [17]:
PREVIOUSBUSINESS.columns = [
    'business_name', 
    'registration_no', 
    'status', 
    'position', 
    'cessation_date', 
]
PREVIOUSBUSINESS

,business_name,registration_no,status,position,cessation_date
0,ABCDE PTE LTD,19960XXXXC,LIVE,DIRECTOR,09/11/19XX
1,NETWORK DENKI,5021XXXXW,TERMINATION,OWNER,21/10/20XX
2,PIONEER INSTRUMENTS,5038XXXXC,CANCELLATION,OWNER,09/11/19XX


Table 5: CURRENT BUSINESS LITIGATION TRACE \nNote: Available Information As Per Experian Database

In [18]:
currentBusLitTraceData = {'Name': [], 'Year': []}
currentBusLitTraceArr = tablesDict['CURRENT BUSINESS LITIGATION TRACE\nNote: Available Information As Per Experian Database']
# Observation tells us that each business should have a total of 7 rows of data
for index in range(0, len(currentBusLitTraceArr), 7):
    for i in range(2, 7):
        currentBusLitTraceData['Name'].append(currentBusLitTraceArr[index][1])
        columnNames = [columnName.split('\n')[0] for columnName in currentBusLitTraceArr[index + 1]]
        currentBusLitTraceData['Year'].append(currentBusLitTraceArr[index + 1][1].split('\n')[1])
        for j in range(3):
            if columnNames[j] in currentBusLitTraceData:
                currentBusLitTraceData[columnNames[j]].append(currentBusLitTraceArr[index + i][j])
            else:
                currentBusLitTraceData[columnNames[j]] = [currentBusLitTraceArr[index + i][j]]
CURRENTBUSINESSLITIGATIONTRACE = pd.DataFrame(currentBusLitTraceData)
CURRENTBUSINESSLITIGATIONTRACE

,Name,Year,Enquiries,Current Year On/After 9 Jan,Previous Year Before 9 Jan
0,ABC HOLDINGS PTE LTD,20XX,"Litigation Trace: Magistrate Court\n(< $60,000)",0,1
1,ABC HOLDINGS PTE LTD,20XX,"Litigation Trace: District Court\n($60,000 - $...",1,0
2,ABC HOLDINGS PTE LTD,20XX,"Litigation Trace: High Court\n(> $250,000)",0,0
3,ABC HOLDINGS PTE LTD,20XX,Winding Up Trace,0,0
4,ABC HOLDINGS PTE LTD,20XX,OS/OA/OP/OM Trace,0,0
5,WWW PTE LTD,20 XX,"Litigation Trace: Magistrate Court\n(< $60,000)",0,1
6,WWW PTE LTD,20 XX,"Litigation Trace: District Court\n($60,000 - $...",1,0
7,WWW PTE LTD,20 XX,"Litigation Trace: High Court\n(> $250,000)",0,0
8,WWW PTE LTD,20 XX,Winding Up Trace,0,0
9,WWW PTE LTD,20 XX,OS/OA/OP/OM Trace,0,0


In [19]:
columnNames = [
    'Name', 
    'Year', 
    'Enquiries', 
    'Current Year On/After 9 Jan',
    'Previous Year Before 9 Jan', 
]
for actual, expected in itertools.zip_longest(CURRENTBUSINESSLITIGATIONTRACE.columns, columnNames):
    assert actual == expected, f'CURRENT BUSINESS LITIGATION TRACE has column {actual} but should be {expected}'

In [20]:
CURRENTBUSINESSLITIGATIONTRACE.columns = [
    'business_name', 
    'year', 
    'enquiries', 
    'current_year_after_9_Jan', 
    'previous_year_before_9_Jan'
]
CURRENTBUSINESSLITIGATIONTRACE

,business_name,year,enquiries,current_year_after_9_Jan,previous_year_before_9_Jan
0,ABC HOLDINGS PTE LTD,20XX,"Litigation Trace: Magistrate Court\n(< $60,000)",0,1
1,ABC HOLDINGS PTE LTD,20XX,"Litigation Trace: District Court\n($60,000 - $...",1,0
2,ABC HOLDINGS PTE LTD,20XX,"Litigation Trace: High Court\n(> $250,000)",0,0
3,ABC HOLDINGS PTE LTD,20XX,Winding Up Trace,0,0
4,ABC HOLDINGS PTE LTD,20XX,OS/OA/OP/OM Trace,0,0
5,WWW PTE LTD,20 XX,"Litigation Trace: Magistrate Court\n(< $60,000)",0,1
6,WWW PTE LTD,20 XX,"Litigation Trace: District Court\n($60,000 - $...",1,0
7,WWW PTE LTD,20 XX,"Litigation Trace: High Court\n(> $250,000)",0,0
8,WWW PTE LTD,20 XX,Winding Up Trace,0,0
9,WWW PTE LTD,20 XX,OS/OA/OP/OM Trace,0,0


Table 6: BUSINESS LITIGATION SEARCH – WRIT OF SUMMONS/ ORIGINATING CLAIM

In [21]:
# The first 6 rows are information on the dataset, which we would not require in our table construction
# Likewise, the last row in the array is irrelevant
businessLitigationSearchArr = tablesDict['BUSINESS LITIGATION SEARCH – WRIT OF SUMMONS/ ORIGINATING CLAIM'][6:-1]

businessLitigationSearchHeaders = [
    'SN', 
    'Court/Type', 
    'Case No.', 
    'Year', 
    'Filing Date', 
    'Plaintiff / \nPetitioner\n/Claimant', 
    'Case Status / \nStatus Date/ \nRemarks', 
    'Plaintiff/ \nClaimant \nStatus/ \nStatus Date', 
    'Defendant \nStatus*/ \nStatus Date', 
    'Amount(s)', 
    'Last Updated in \nExperian \nDatabase',
    'Main Cause'
]

# Initialise the dataframe 'Business Litigation Search'
BUSINESSLITIGATIONSEARCH = pd.DataFrame(columns=[h.replace('\n', '').replace('*','') for h in businessLitigationSearchHeaders])

# Concatenate the data and append them to the dataframe
businessLitigationSearchData = businessLitigationSearchArr[1:]
for rowIndex in range(0, len(businessLitigationSearchData), 2):
    rowData = businessLitigationSearchData[rowIndex]
    rowData.append(businessLitigationSearchData[rowIndex+1][-1])
   
    BUSINESSLITIGATIONSEARCH = BUSINESSLITIGATIONSEARCH.append(
            pd.DataFrame([rowData], columns=list(BUSINESSLITIGATIONSEARCH.columns)), ignore_index=True)

# Split Case Status, Date and Remarks into separate columns
BUSINESSLITIGATIONSEARCH[['Case Status', 'Case Status Date', 'Case Status Remarks']] = BUSINESSLITIGATIONSEARCH['Case Status / Status Date/ Remarks'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:3])

# Split Plaintiff / Claimant Status and Date into separate columns
BUSINESSLITIGATIONSEARCH[['Plaintiff / Claimant Status', 'Plaintiff / Claimant Status Date']] = BUSINESSLITIGATIONSEARCH['Plaintiff/ Claimant Status/ Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2])

# Split Defendant Status and Date into separate columns
BUSINESSLITIGATIONSEARCH[['Defendant Status', 'Defendant Status Date']] = BUSINESSLITIGATIONSEARCH['Defendant Status/ Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2])
BUSINESSLITIGATIONSEARCH = BUSINESSLITIGATIONSEARCH[[
    'SN', 
    'Court/Type', 
    'Case No.', 
    'Year', 
    'Filing Date', 
    'Plaintiff / Petitioner/Claimant', 
    'Case Status',
    'Case Status Date', 
    'Case Status Remarks',
    'Plaintiff / Claimant Status', 
    'Plaintiff / Claimant Status Date',
    'Defendant Status', 
    'Defendant Status Date',
    'Amount(s)', 
    'Last Updated in Experian Database',
    'Main Cause'
]]

BUSINESSLITIGATIONSEARCH

,SN,Court/Type,Case No.,Year,Filing Date,Plaintiff / Petitioner/Claimant,Case Status,Case Status Date,Case Status Remarks,Plaintiff / Claimant Status,Plaintiff / Claimant Status Date,Defendant Status,Defendant Status Date,Amount(s),Last Updated in Experian Database,Main Cause
0,1,MC/ WOS,106,20XX,12/09/20XX,TRANS\nPTE LTD,PENDING,08/03/20XX,,PENDING,08/03/20XX,PENDING,08/03/20XX,"SGD\n50,000.00",15/08/20XX,CONTRACTS - GENERAL
1,2,DC/ OC,17,20XX,13/04/20XX,SAHA\nCHANDRA,PENDING,20/04/20XX,,PENDING,13/04/20XX,PENDING,13/04/20XX,-,20/04/20XX,CONTRACTS – CREDIT FACILITIES /OVERDRAFT/GURAN...


In [22]:
columnNames = [
    'SN', 
    'Court/Type', 
    'Case No.', 
    'Year', 
    'Filing Date',
    'Plaintiff / Petitioner/Claimant', 
    'Case Status', 
    'Case Status Date',
    'Case Status Remarks', 
    'Plaintiff / Claimant Status',
    'Plaintiff / Claimant Status Date', 
    'Defendant Status',
    'Defendant Status Date', 
    'Amount(s)',
    'Last Updated in Experian Database', 
    'Main Cause', 
]
for actual, expected in itertools.zip_longest(BUSINESSLITIGATIONSEARCH.columns, columnNames):
    assert actual == expected, f'BUSINESS LITIGATION SEARCH has column {actual} but should be {expected}'

In [23]:
BUSINESSLITIGATIONSEARCH.columns = [
    'sn', 
    'court_type', 
    'case_no', 
    'year', 
    'filing_date', 
    'plaintiff', 
    'case_status', 
    'case_status_date', 
    'case_status_remarks', 
    'plaintiff_status', 
    'plaintiff_status_date', 
    'defendant_status', 
    'defendant_status_date', 
    'amount', 
    'last_update', 
    'main_cause', 
]
BUSINESSLITIGATIONSEARCH

,sn,court_type,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,plaintiff_status,plaintiff_status_date,defendant_status,defendant_status_date,amount,last_update,main_cause
0,1,MC/ WOS,106,20XX,12/09/20XX,TRANS\nPTE LTD,PENDING,08/03/20XX,,PENDING,08/03/20XX,PENDING,08/03/20XX,"SGD\n50,000.00",15/08/20XX,CONTRACTS - GENERAL
1,2,DC/ OC,17,20XX,13/04/20XX,SAHA\nCHANDRA,PENDING,20/04/20XX,,PENDING,13/04/20XX,PENDING,13/04/20XX,-,20/04/20XX,CONTRACTS – CREDIT FACILITIES /OVERDRAFT/GURAN...


In [24]:
clustering_model = joblib.load('../../Litigation Clustering/litigation_main_cause_clustering.pkl')

BUSINESSLITIGATIONSEARCH['main_cause_category'] = BUSINESSLITIGATIONSEARCH['main_cause'].apply(lambda x: label_litigation_main_cause(clustering_model, x))

clustering_model = joblib.load('../../Litigation Clustering/litigation_case_status_clustering.pkl')

BUSINESSLITIGATIONSEARCH['case_status_category'] = BUSINESSLITIGATIONSEARCH['case_status'].apply(lambda x: label_litigation_case_status(clustering_model, x))

clustering_model = joblib.load('../../Litigation Clustering/litigation_party_status_clustering.pkl')

BUSINESSLITIGATIONSEARCH['plaintiff_status_category'] = BUSINESSLITIGATIONSEARCH['plaintiff_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))
BUSINESSLITIGATIONSEARCH['defendant_status_category'] = BUSINESSLITIGATIONSEARCH['defendant_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))

BUSINESSLITIGATIONSEARCH

,sn,court_type,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,plaintiff_status,plaintiff_status_date,defendant_status,defendant_status_date,amount,last_update,main_cause,main_cause_category,case_status_category,plaintiff_status_category,defendant_status_category
0,1,MC/ WOS,106,20XX,12/09/20XX,TRANS\nPTE LTD,PENDING,08/03/20XX,,PENDING,08/03/20XX,PENDING,08/03/20XX,"SGD\n50,000.00",15/08/20XX,CONTRACTS - GENERAL,Contracts,Others,Others,Others
1,2,DC/ OC,17,20XX,13/04/20XX,SAHA\nCHANDRA,PENDING,20/04/20XX,,PENDING,13/04/20XX,PENDING,13/04/20XX,-,20/04/20XX,CONTRACTS – CREDIT FACILITIES /OVERDRAFT/GURAN...,Financials,Others,Others,Others


Table 7: BANKRUPTCY RECORD(S)

In [25]:
# We can ignore the last row of the array, as the information is irrelevant
# The first row in the array are the headers for the table, hence the main data only starts from row 2 onwards
bankruptcyRecordsArr = tablesDict['BANKRUPTCY RECORD(S)'][2:-1]

bankruptcyRecordsHeaders = [
   'Case No.', 
   'Year', 
   'Filing Date', 
   'Plaintiff / \nPetitioner/ \nClaimant', 
   'Case Status/ \nStatus Date/ \nRemark', 
   'Plaintiff/ \nClaimant \nStatus/ \nStatus Date', 
   'Defendant \nStatus/ \nStatus Date', 
   'Amount(s)', 
   'Last Updated \nin Experian \nDatabase',
   'Name of Private Trustees \nAppointed (As per \nExperian Database)',
]

# Initialise the dataframe 'Bankruptcy Records'
BANKRUPTCYRECORDS = pd.DataFrame(columns=[h.replace('\n', '').replace(':','') for h in bankruptcyRecordsHeaders])

# Concat the data and append them to the dataframe
bankruptcyRecordsData = bankruptcyRecordsArr[1:]
for rowIndex in range(0, len(bankruptcyRecordsData), 2):
    rowData = bankruptcyRecordsData[rowIndex]
    rowData.append(bankruptcyRecordsData[rowIndex+1][-1])
    BANKRUPTCYRECORDS = BANKRUPTCYRECORDS.append(
            pd.DataFrame([rowData], columns=list(BANKRUPTCYRECORDS.columns)), ignore_index=True)

# Split Case Status, Date and Remarks into separate columns
BANKRUPTCYRECORDS[['Case Status', 'Case Status Date', 'Case Status Remarks']] = BANKRUPTCYRECORDS['Case Status/ Status Date/ Remark'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:3])

# Split Plaintiff / Claimant Status and Date into separate columns
BANKRUPTCYRECORDS[['Plaintiff / Claimant Status', 'Plaintiff / Claimant Status Date']] = BANKRUPTCYRECORDS['Plaintiff/ Claimant Status/ Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2])

# Split Defendant Status and Date into separate columns
BANKRUPTCYRECORDS[['Defendant Status', 'Defendant Status Date']] = BANKRUPTCYRECORDS['Defendant Status/ Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2])
BANKRUPTCYRECORDS = BANKRUPTCYRECORDS[[
   'Case No.', 
   'Year', 
   'Filing Date', 
   'Plaintiff / Petitioner/ Claimant',
   'Case Status',
   'Case Status Date', 
   'Case Status Remarks',
   'Plaintiff / Claimant Status', 
   'Plaintiff / Claimant Status Date',
   'Defendant Status', 
   'Defendant Status Date',
   'Amount(s)', 
   'Last Updated in Experian Database',
   'Name of Private Trustees Appointed (As per Experian Database)'
]]
BANKRUPTCYRECORDS

,Case No.,Year,Filing Date,Plaintiff / Petitioner/ Claimant,Case Status,Case Status Date,Case Status Remarks,Plaintiff / Claimant Status,Plaintiff / Claimant Status Date,Defendant Status,Defendant Status Date,Amount(s),Last Updated in Experian Database,Name of Private Trustees Appointed (As per Experian Database)
0,126,20XX,29/09/20XX,ONE BANK LTD,WITHDRAWN,03/08/20XX,,WITHDRAWN,03/08/20XX,DISCHARGE,03/08/20XX,"SGD 50,000.00",15/01/20XX,TAN EE EE\nABC PARTNER LLC\n5 SHENTON WAY #01-...


In [26]:
columnNames = [
    'Case No.', 
    'Year', 
    'Filing Date', 
    'Plaintiff / Petitioner/ Claimant',
    'Case Status', 
    'Case Status Date', 
    'Case Status Remarks',
    'Plaintiff / Claimant Status', 
    'Plaintiff / Claimant Status Date',
    'Defendant Status', 
    'Defendant Status Date', 
    'Amount(s)',
    'Last Updated in Experian Database',
    'Name of Private Trustees Appointed (As per Experian Database)', 
]
for actual, expected in itertools.zip_longest(BANKRUPTCYRECORDS.columns, columnNames):
    assert actual == expected, f'BANKRUPTCY RECORDS SEARCH has column {actual} but should be {expected}'

In [27]:
BANKRUPTCYRECORDS.columns = [
    'case_no', 
    'year', 
    'filing_date', 
    'plaintiff', 
    'case_status', 
    'case_status_date', 
    'case_status_remarks', 
    'plaintiff_status', 
    'plaintiff_status_date', 
    'defendant_status', 
    'defendant_status_date', 
    'amount', 
    'last_update', 
    'name_of_private_trustees', 
]
BANKRUPTCYRECORDS

,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,plaintiff_status,plaintiff_status_date,defendant_status,defendant_status_date,amount,last_update,name_of_private_trustees
0,126,20XX,29/09/20XX,ONE BANK LTD,WITHDRAWN,03/08/20XX,,WITHDRAWN,03/08/20XX,DISCHARGE,03/08/20XX,"SGD 50,000.00",15/01/20XX,TAN EE EE\nABC PARTNER LLC\n5 SHENTON WAY #01-...


In [28]:
clustering_model = joblib.load('../../Litigation Clustering/litigation_case_status_clustering.pkl')

BANKRUPTCYRECORDS['case_status_category'] = BANKRUPTCYRECORDS['case_status'].apply(lambda x: label_litigation_case_status(clustering_model, x))

clustering_model = joblib.load('../../Litigation Clustering/litigation_party_status_clustering.pkl')

BANKRUPTCYRECORDS['plaintiff_status_category'] = BANKRUPTCYRECORDS['plaintiff_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))
BANKRUPTCYRECORDS['defendant_status_category'] = BANKRUPTCYRECORDS['defendant_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))

BANKRUPTCYRECORDS

,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,plaintiff_status,plaintiff_status_date,defendant_status,defendant_status_date,amount,last_update,name_of_private_trustees,case_status_category,plaintiff_status_category,defendant_status_category
0,126,20XX,29/09/20XX,ONE BANK LTD,WITHDRAWN,03/08/20XX,,WITHDRAWN,03/08/20XX,DISCHARGE,03/08/20XX,"SGD 50,000.00",15/01/20XX,TAN EE EE\nABC PARTNER LLC\n5 SHENTON WAY #01-...,Others,Others,Discharge Notice


Table 8: INDIVIDUAL LITIGATION SEARCH – WRIT OF SUMMONS/ ORIGINATING CLAIM

In [29]:
# Rows 1 to 5 are information related to table details. Hence, we can exclude them from the table construction
# Likewise, the last row is irrelevant
indvLitigationSearchArr = tablesDict['INDIVIDUAL LITIGATION SEARCH – WRIT OF SUMMONS/ ORIGINATING CLAIM'][5:-1]

indvLitigationSearchHeaders = [
   'SN', 
   'Court/ Type', 
   'Case \nNo.', 
   'Year', 
   'Filing \nDate', 
   'Plaintiff/ \nPetitioner\n/Claimant', 
   'Case Status / \nStatus Date \nRemarks', 
   'Plaintiff/ \nClaimant \nStatus/ \nStatus Date',
   'Defendant \nStatus/ \n \nStatus Date', 
   'Amount(s)', 
   'Last Updated \nin Experian \nDatabase',
   'Main Cause',
]

# Initialise the dataframe
INDIVIDUALLITIGATIONSEARCH = pd.DataFrame(columns=[h.replace('\n', '').replace(':','') for h in indvLitigationSearchHeaders])

# Concat the data and append them to the dataframe
indvLitigationSearchData = indvLitigationSearchArr[1:]
for rowIndex in range(0, len(indvLitigationSearchData), 2):

   # Each data set has 2 rows
   rowData = indvLitigationSearchData[rowIndex]
   rowData.append(indvLitigationSearchData[rowIndex+1][-1])
   INDIVIDUALLITIGATIONSEARCH = INDIVIDUALLITIGATIONSEARCH.append(
      pd.DataFrame([rowData], columns=list(INDIVIDUALLITIGATIONSEARCH.columns)), ignore_index=True)

# Split Case Status, Date and Remarks into separate columns
INDIVIDUALLITIGATIONSEARCH[['Case Status', 'Case Status Date', 'Case Status Remarks']] = INDIVIDUALLITIGATIONSEARCH['Case Status / Status Date Remarks'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:3])

# Split Plaintiff / Claimant Status and Date into separate columns
INDIVIDUALLITIGATIONSEARCH[['Plaintiff / Claimant Status', 'Plaintiff / Claimant Status Date']] = INDIVIDUALLITIGATIONSEARCH['Plaintiff/ Claimant Status/ Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2])

# Split Defendant Status and Date into separate columns
INDIVIDUALLITIGATIONSEARCH[['Defendant Status', 'Defendant Status Date']] = INDIVIDUALLITIGATIONSEARCH['Defendant Status/  Status Date'].apply(lambda x: pd.Series(re.split('(../....)', str(x).replace('\n', ''), 1))[:2])

INDIVIDUALLITIGATIONSEARCH = INDIVIDUALLITIGATIONSEARCH[[
   'SN', 
   'Court/ Type', 
   'Case No.', 
   'Year', 
   'Filing Date', 
   'Plaintiff/ Petitioner/Claimant', 
   'Case Status',
   'Case Status Date', 
   'Case Status Remarks',
   'Plaintiff / Claimant Status', 
   'Plaintiff / Claimant Status Date',
   'Defendant Status', 
   'Defendant Status Date',
   'Amount(s)', 
   'Last Updated in Experian Database',
   'Main Cause',
]]

INDIVIDUALLITIGATIONSEARCH

,SN,Court/ Type,Case No.,Year,Filing Date,Plaintiff/ Petitioner/Claimant,Case Status,Case Status Date,Case Status Remarks,Plaintiff / Claimant Status,Plaintiff / Claimant Status Date,Defendant Status,Defendant Status Date,Amount(s),Last Updated in Experian Database,Main Cause
0,1,SC/ WOS,33,20XX,13/03/20XX,TAN SENG\nHONG,PENDINGTRANSFERREDFROM DC106/20XX,NaN,NaN,PENDING,13/03/20XX,PENDING,13/03/20XX,"1) SGD 57,330.00\n2) SGD 45,000.00",21/03/20XX,CONTRACTS - GENERAL
1,2,DC/ WOS,106,20XX,12/09/20XX,TAN SENG\nHONG,TRANSFERRED,13/03/20XX,TRANSFERRED TOHC 33/20XX,-,NaN,-,NaN,"1) SGD 50,000.00",20/10/20XX,CONTRACTS - GENERAL
2,3,DC/ OC,106,20XX,13/04/20XX,SAHA\nCHANDRA,PENDING,20/04/20XX,,PENDING,13/04/20XX,PENDING,13/04/20XX,-,22/04/20XX,CONTRACTS - GENERAL


In [30]:
columnNames = [
    'SN', 
    'Court/ Type', 
    'Case No.', 
    'Year', 
    'Filing Date',
    'Plaintiff/ Petitioner/Claimant', 
    'Case Status', 
    'Case Status Date',
    'Case Status Remarks', 
    'Plaintiff / Claimant Status',
    'Plaintiff / Claimant Status Date', 
    'Defendant Status',
    'Defendant Status Date', 
    'Amount(s)',
    'Last Updated in Experian Database', 
    'Main Cause',
]
for actual, expected in itertools.zip_longest(INDIVIDUALLITIGATIONSEARCH.columns, columnNames):
    assert actual == expected, f'INDIVIDUAL LITIGATION SEARCH has column {actual} but should be {expected}'


In [31]:
INDIVIDUALLITIGATIONSEARCH.columns = [
    'sn', 
    'type', 
    'case_no', 
    'year', 
    'filing_date', 
    'plaintiff', 
    'case_status', 
    'case_status_date', 
    'case_status_remarks', 
    'plaintiff_status', 
    'plaintiff_status_date', 
    'defendant_status', 
    'defendant_status_date', 
    'amount', 
    'last_update', 
    'main_cause', 
]
INDIVIDUALLITIGATIONSEARCH

,sn,type,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,plaintiff_status,plaintiff_status_date,defendant_status,defendant_status_date,amount,last_update,main_cause
0,1,SC/ WOS,33,20XX,13/03/20XX,TAN SENG\nHONG,PENDINGTRANSFERREDFROM DC106/20XX,NaN,NaN,PENDING,13/03/20XX,PENDING,13/03/20XX,"1) SGD 57,330.00\n2) SGD 45,000.00",21/03/20XX,CONTRACTS - GENERAL
1,2,DC/ WOS,106,20XX,12/09/20XX,TAN SENG\nHONG,TRANSFERRED,13/03/20XX,TRANSFERRED TOHC 33/20XX,-,NaN,-,NaN,"1) SGD 50,000.00",20/10/20XX,CONTRACTS - GENERAL
2,3,DC/ OC,106,20XX,13/04/20XX,SAHA\nCHANDRA,PENDING,20/04/20XX,,PENDING,13/04/20XX,PENDING,13/04/20XX,-,22/04/20XX,CONTRACTS - GENERAL


In [32]:
clustering_model = joblib.load('../../Litigation Clustering/litigation_main_cause_clustering.pkl')

INDIVIDUALLITIGATIONSEARCH['main_cause_category'] = INDIVIDUALLITIGATIONSEARCH['main_cause'].apply(lambda x: label_litigation_main_cause(clustering_model, x))

clustering_model = joblib.load('../../Litigation Clustering/litigation_case_status_clustering.pkl')

INDIVIDUALLITIGATIONSEARCH['case_status_category'] = INDIVIDUALLITIGATIONSEARCH['case_status'].apply(lambda x: label_litigation_case_status(clustering_model, x))

clustering_model = joblib.load('../../Litigation Clustering/litigation_party_status_clustering.pkl')

INDIVIDUALLITIGATIONSEARCH['plaintiff_status_category'] = INDIVIDUALLITIGATIONSEARCH['plaintiff_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))
INDIVIDUALLITIGATIONSEARCH['defendant_status_category'] = INDIVIDUALLITIGATIONSEARCH['defendant_status'].apply(lambda x: label_litigation_party_status(clustering_model, x))

INDIVIDUALLITIGATIONSEARCH

,sn,type,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,plaintiff_status,plaintiff_status_date,defendant_status,defendant_status_date,amount,last_update,main_cause,main_cause_category,case_status_category,plaintiff_status_category,defendant_status_category
0,1,SC/ WOS,33,20XX,13/03/20XX,TAN SENG\nHONG,PENDINGTRANSFERREDFROM DC106/20XX,NaN,NaN,PENDING,13/03/20XX,PENDING,13/03/20XX,"1) SGD 57,330.00\n2) SGD 45,000.00",21/03/20XX,CONTRACTS - GENERAL,Contracts,Others,Others,Others
1,2,DC/ WOS,106,20XX,12/09/20XX,TAN SENG\nHONG,TRANSFERRED,13/03/20XX,TRANSFERRED TOHC 33/20XX,-,NaN,-,NaN,"1) SGD 50,000.00",20/10/20XX,CONTRACTS - GENERAL,Contracts,Others,Others,Others
2,3,DC/ OC,106,20XX,13/04/20XX,SAHA\nCHANDRA,PENDING,20/04/20XX,,PENDING,13/04/20XX,PENDING,13/04/20XX,-,22/04/20XX,CONTRACTS - GENERAL,Contracts,Others,Others,Others


Table 9: ACCREDITATIONS – CURRENT BUSINESS

In [33]:
# Initialise the dataframe
accreditationData = {'Business': [], 'Ranking Source': [], 'Rank': [], }

for data in tablesDict['ACCREDITATIONS – CURRENT BUSINESS']:
    if len(data) > 1:
        accreditationData['Ranking Source'].append(data[0])
        accreditationData['Rank'].append(data[1])
    else:
        accreditationData['Business'].append(data[0])

ACCREDITATIONS = pd.DataFrame(accreditationData)
ACCREDITATIONS

,Business,Ranking Source,Rank
0,ABC HOLDINGS PTE LTD,Singapore1000/SME1000 Rankings,2008 S1000: Ranked 1456
1,WWW PTE LTD,Singapore1000/SME1000 Rankings,2008 S1000: Ranked 200
2,T.A.K. SPECIALIST,Singapore1000/SME1000 Rankings,-


In [34]:
columnNames = [
    'Business', 
    'Ranking Source', 
    'Rank', 
]
for actual, expected in itertools.zip_longest(ACCREDITATIONS.columns, columnNames):
    assert actual == expected, f'ACCREDITATIONS has column {actual} but should be {expected}'

In [35]:
ACCREDITATIONS.columns = [
    'business', 
    'award_name', 
    'award_ranking', 
]
ACCREDITATIONS

,business,award_name,award_ranking
0,ABC HOLDINGS PTE LTD,Singapore1000/SME1000 Rankings,2008 S1000: Ranked 1456
1,WWW PTE LTD,Singapore1000/SME1000 Rankings,2008 S1000: Ranked 200
2,T.A.K. SPECIALIST,Singapore1000/SME1000 Rankings,-


Table 10: FINANCIAL RELATED SEARCH COUNT

In [36]:
financialRelatedSearchCountData = {
    'year': [], 
    'month': [], 
    'purpose': [], 
    'count': [], 
}

def add(year, month, purpose, count):
    financialRelatedSearchCountData['year'].append(year)
    financialRelatedSearchCountData['month'].append(month)
    financialRelatedSearchCountData['purpose'].append(purpose)
    financialRelatedSearchCountData['count'].append(count)

for data in tablesDict['FINANCIAL RELATED SEARCH COUNT'][2:]:
    if len(data) == 14:
        year = data[0]
        add(year, 'All', 'All', data[1])
        for index in range(2, 14):
            add(year, tablesDict['FINANCIAL RELATED SEARCH COUNT'][1][index], 'All', data[index])
    else:
        for index in range(1, 13):
            add(year, tablesDict['FINANCIAL RELATED SEARCH COUNT'][1][index], data[0], data[index])
FINRELSEARCHCOUNT = pd.DataFrame(financialRelatedSearchCountData)
FINRELSEARCHCOUNT

,year,month,purpose,count
0,20XX,All,All,3
1,20XX,Jan,All,3
2,20XX,Feb,All,0
3,20XX,Mar,All,0
4,20XX,Apr,All,0
...,...,...,...,...
58,20XX,Aug,All,0
59,20XX,Sep,All,0
60,20XX,Oct,All,0
61,20XX,Nov,All,0


Table 11: COMMERCIAL RELATED SEARCH COUNT

In [37]:
commercialRelatedSearchCountData = {
    'year': [], 
    'month': [], 
    'purpose': [], 
    'count': [], 
}

def add(year, month, purpose, count):
    commercialRelatedSearchCountData['year'].append(year)
    commercialRelatedSearchCountData['month'].append(month)
    commercialRelatedSearchCountData['purpose'].append(purpose)
    commercialRelatedSearchCountData['count'].append(count)

for data in tablesDict['COMMERCIAL RELATED SEARCH COUNT'][2:]:
    if len(data) == 14:
        year = data[0]
        add(year, 'All', 'All', data[1])
        for index in range(2, 14):
            add(year, tablesDict['COMMERCIAL RELATED SEARCH COUNT'][1][index], 'All', data[index])
    else:
        for index in range(1, 13):
            add(year, tablesDict['COMMERCIAL RELATED SEARCH COUNT'][1][index], data[0], data[index])
COMRELSEARCHCOUNT = pd.DataFrame(commercialRelatedSearchCountData)
COMRELSEARCHCOUNT

,year,month,purpose,count
0,20XX,All,All,1
1,20XX,Jan,All,1
2,20XX,Feb,All,0
3,20XX,Mar,All,0
4,20XX,Apr,All,0
5,20XX,May,All,0
6,20XX,Jun,All,0
7,20XX,Jul,All,0
8,20XX,Aug,All,0
9,20XX,Sep,All,0


Table 12: LEGAL RELATED SEARCH COUNT

In [38]:
legalRelatedSearchCountData = {
    'year': [], 
    'month': [], 
    'purpose': [], 
    'count': [], 
}

def add(year, month, purpose, count):
    legalRelatedSearchCountData['year'].append(year)
    legalRelatedSearchCountData['month'].append(month)
    legalRelatedSearchCountData['purpose'].append(purpose)
    legalRelatedSearchCountData['count'].append(count)

for data in tablesDict['LEGAL RELATED SEARCH COUNT'][2:-5]:
    if len(data) == 14:
        year = data[0]
        add(year, 'All', 'All', data[1])
        for index in range(2, 14):
            add(year, tablesDict['LEGAL RELATED SEARCH COUNT'][1][index], 'All', data[index])
    else:
        for index in range(1, len(data)):
            add(year, tablesDict['LEGAL RELATED SEARCH COUNT'][1][index], data[0], data[index])
LEGRELSEARCHCOUNT = pd.DataFrame(legalRelatedSearchCountData)
LEGRELSEARCHCOUNT

,year,month,purpose,count
0,20XX,All,All,2
1,20XX,Jan,All,2
2,20XX,Feb,All,0
3,20XX,Mar,All,0
4,20XX,Apr,All,0
5,20XX,May,All,0
6,20XX,Jun,All,0
7,20XX,Jul,All,0
8,20XX,Aug,All,0
9,20XX,Sep,All,0


## Export all the tables to excel / csv

In [73]:
# Output Data
writer = pd.ExcelWriter(f'../Output/tables_v1.xlsx', engine='xlsxwriter')

REQUESTCRITERIA.to_excel(writer, sheet_name='Request Criteria')
SEARCHRECORD.to_excel(writer, sheet_name='Search Record')
CURRENTBUSINESS.to_excel(writer, sheet_name='Current Business')
PREVIOUSBUSINESS.to_excel(writer, sheet_name='Previous Business')
CURRENTBUSINESSLITIGATIONTRACE.to_excel(writer, sheet_name='Current Business Lit Trace')
BUSINESSLITIGATIONSEARCH.to_excel(writer, sheet_name='Business Lit Search')
BANKRUPTCYRECORDS.to_excel(writer, sheet_name='Bankruptcy Records')
INDIVIDUALLITIGATIONSEARCH.to_excel(writer, sheet_name='Individual Lit Search')
ACCREDITATIONS.to_excel(writer, sheet_name='Accreditation')
FINRELSEARCHCOUNT.to_excel(writer, sheet_name='Financial Related Search Count')
COMRELSEARCHCOUNT.to_excel(writer, sheet_name='Commercial Related Search Count')
LEGRELSEARCHCOUNT.to_excel(writer, sheet_name='Legal Related Search Count')

writer.close()